In [1]:
import os
os.chdir("..")
print(os.getcwd())

c:\Users\NASA2004\OneDrive\Documents\transformer-from-scratch


In [2]:
# scaled dot-product attention

import torch
import torch.nn.functional as F
import math

def scaled_dot_product_attention(Q, K, V, mask=None):

    d_k = Q.size(-1)

    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)

    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))

    attn_weights = F.softmax(scores, dim=-1)

    output = torch.matmul(attn_weights, V)

    return output, attn_weights

In [3]:
# verifying

Q = torch.randn(2, 4, 10, 16)
K = torch.randn(2, 4, 10, 16)
V = torch.randn(2, 4, 10, 16)

out, attn = scaled_dot_product_attention(Q, K, V)
print(out.shape, attn.shape)

torch.Size([2, 4, 10, 16]) torch.Size([2, 4, 10, 10])


In [4]:
# multi-head attention

import torch.nn as nn

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        batch_size, seq_len, d_model = x.size()
        x = x.view(batch_size, seq_len, self.num_heads, self.d_k)
        return x.transpose(1, 2)

    def combine_heads(self, x):
        batch_size, num_heads, seq_len, d_k = x.size()
        x = x.transpose(2, 1)
        return x.contiguous().view(batch_size, seq_len, self.d_model)

    def forward(self, Q, K, V, mask=None):
        Q = self.split_heads(self.W_Q(Q))
        K = self.split_heads(self.W_K(K))
        V = self.split_heads(self.W_V(V))

        attn_output, attn_weights = scaled_dot_product_attention(Q, K, V, mask)

        output = self.W_O(self.combine_heads(attn_output))
        return output, attn_weights

In [5]:
# verifying

d_model, num_heads = 512, 8
mha = MultiHeadAttention(d_model, num_heads)

x = torch.randn(2, 10, d_model)
out, attn = mha(x, x, x)
print(out.shape, attn.shape)

torch.Size([2, 10, 512]) torch.Size([2, 8, 10, 10])


In [6]:
# testing the import from src

from src.attention import MultiHeadAttention

mha = MultiHeadAttention(d_model=512, num_heads=8)
x = torch.randn(2, 10, 512)
out, attn = mha(x, x, x)
print(out.shape, attn.shape)

torch.Size([2, 10, 512]) torch.Size([2, 8, 10, 10])


In [7]:
# checking that masking is actually working

seq_len = 5
batch_size = 1
d_model, num_heads = 512, 8

x = torch.randn(batch_size, seq_len, d_model)

mask = torch.ones(batch_size, 1, seq_len, seq_len)
mask[:, :, :, 3:] = 0

mha = MultiHeadAttention(d_model, num_heads)
out, attn = mha(x, x, x, mask=mask)

print(attn[0, 0])

tensor([[0.3837, 0.2180, 0.3983, 0.0000, 0.0000],
        [0.4072, 0.3376, 0.2552, 0.0000, 0.0000],
        [0.2955, 0.3047, 0.3999, 0.0000, 0.0000],
        [0.3019, 0.3610, 0.3371, 0.0000, 0.0000],
        [0.3785, 0.3382, 0.2832, 0.0000, 0.0000]], grad_fn=<SelectBackward0>)
